In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, io, viz, tuning
from fraud_detect.models import ModelBackend

warnings.filterwarnings("ignore")
viz.configure_style()

## 11. Hyperparameter Tuning with Optuna

Optimise LightGBM, XGBoost, and CatBoost hyperparameters.
Default params from notebook 10 are the baseline; Optuna
searches for better configurations.

> Search spaces in `fraud_detect.config` | Logic in `fraud_detect.tuning`

In [ ]:
train = io.load_train_features()
print(f"Data loaded: {train.shape}")
print(f"Fraud rate : {train['isFraud'].mean()*100:.2f}%")

### 11.1 LightGBM Tuning

In [ ]:
lgb_space = tuning.build_search_space(ModelBackend.LIGHTGBM)
pd.DataFrame([{"Parameter": k, "Low": v["low"], "High": v["high"],
               "Log Scale": v.get("log", False)} for k, v in lgb_space.items()])

In [ ]:
study_lgb = tuning.tune_model(train, backend=ModelBackend.LIGHTGBM,
                              n_trials=config.OPTUNA_N_TRIALS)
if study_lgb is not None:
    tuning.save_best_params(ModelBackend.LIGHTGBM, study_lgb.best_params)

In [ ]:
if study_lgb is not None:
    import optuna
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    trials_df = study_lgb.trials_dataframe()
    axes[0].plot(trials_df["number"], trials_df["value"], "o-", alpha=0.6, markersize=3)
    axes[0].axhline(study_lgb.best_value, color="red", linestyle="--",
                    label=f"Best: {study_lgb.best_value:.5f}")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("Validation AUC")
    axes[0].set_title("LightGBM — Optimisation History")
    axes[0].legend()
    optuna.visualization.matplotlib.plot_param_importances(study_lgb, ax=axes[1])
    axes[1].set_title("LightGBM — Hyperparameter Importance")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "lgbm_tuning_history.png")
    plt.show()
    print(f"Best AUC: {study_lgb.best_value:.5f} | Params: {study_lgb.best_params}")

### 11.2 XGBoost Tuning

In [ ]:
xgb_space = tuning.build_search_space(ModelBackend.XGBOOST)
pd.DataFrame([{"Parameter": k, "Low": v["low"], "High": v["high"],
               "Log Scale": v.get("log", False)} for k, v in xgb_space.items()])

In [ ]:
study_xgb = tuning.tune_model(train, backend=ModelBackend.XGBOOST,
                              n_trials=config.OPTUNA_N_TRIALS)
if study_xgb is not None:
    tuning.save_best_params(ModelBackend.XGBOOST, study_xgb.best_params)

In [ ]:
if study_xgb is not None:
    import optuna
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    trials_df = study_xgb.trials_dataframe()
    axes[0].plot(trials_df["number"], trials_df["value"], "o-", alpha=0.6, markersize=3)
    axes[0].axhline(study_xgb.best_value, color="red", linestyle="--",
                    label=f"Best: {study_xgb.best_value:.5f}")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("Validation AUC")
    axes[0].set_title("XGBoost — Optimisation History")
    axes[0].legend()
    optuna.visualization.matplotlib.plot_param_importances(study_xgb, ax=axes[1])
    axes[1].set_title("XGBoost — Hyperparameter Importance")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "xgb_tuning_history.png")
    plt.show()
    print(f"Best AUC: {study_xgb.best_value:.5f} | Params: {study_xgb.best_params}")

### 11.3 CatBoost Tuning

In [ ]:
cb_space = tuning.build_search_space(ModelBackend.CATBOOST)
pd.DataFrame([{"Parameter": k, "Low": v["low"], "High": v["high"],
               "Log Scale": v.get("log", False)} for k, v in cb_space.items()])

In [ ]:
study_cb = tuning.tune_model(train, backend=ModelBackend.CATBOOST,
                             n_trials=config.OPTUNA_N_TRIALS)
if study_cb is not None:
    tuning.save_best_params(ModelBackend.CATBOOST, study_cb.best_params)

In [ ]:
if study_cb is not None:
    import optuna
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    trials_df = study_cb.trials_dataframe()
    axes[0].plot(trials_df["number"], trials_df["value"], "o-", alpha=0.6, markersize=3)
    axes[0].axhline(study_cb.best_value, color="red", linestyle="--",
                    label=f"Best: {study_cb.best_value:.5f}")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("Validation AUC")
    axes[0].set_title("CatBoost — Optimisation History")
    axes[0].legend()
    optuna.visualization.matplotlib.plot_param_importances(study_cb, ax=axes[1])
    axes[1].set_title("CatBoost — Hyperparameter Importance")
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "cb_tuning_history.png")
    plt.show()
    print(f"Best AUC: {study_cb.best_value:.5f} | Params: {study_cb.best_params}")

### 11.4 Tuning Comparison

In [ ]:
tuning_results = []
for name, study in [("LightGBM", study_lgb), ("XGBoost", study_xgb), ("CatBoost", study_cb)]:
    if study is not None:
        tuning_results.append({"Model": name, "Best Val AUC": f"{study.best_value:.5f}",
                              "Trials": len(study.trials)})
pd.DataFrame(tuning_results)

In [ ]:
for backend in ModelBackend:
    params = tuning.load_best_params(backend)
    print(f"{backend.value.upper()}")
    for k, v in sorted(params.items()):
        print(f"  {k}: {v}")
    print()

### 11.5 Summary

Best params saved to `data/metadata/` for reuse in ensemble & evaluation notebooks.

Next: notebook 12 — ensemble methods.